In [24]:
# pyright: reportOperatorIssue=false

In [25]:
import sympy as sp
from sympy.printing.latex import LatexPrinter
from IPython.display import display, Math

In [26]:
# Main parameters
dx = sp.Symbol('dx', positive=True, real=True)
dt = sp.Symbol('dt', positive=True, real=True)
dy = sp.Symbol('dy', positive=True, real=True)
a = sp.Symbol('a', positive=True, real=True)
b = sp.Symbol('b', positive=True, real=True)
sigma = sp.Symbol('sigma', real=True)

symbol_names = {
    dx: r"\Delta x", 
    dy: r"\Delta y", 
    dt: r"\Delta t", 
    sigma: r"\sigma"
}


In [ ]:
def get_index_str(val, var):
    if val == 0: return var
    elif val > 0: return f"{var}+{val}"
    else: return f"{var}{val}" # val is negative, includes minus sign naturally

u_dict = {}
points = [(i, j) for i in range(-2, 3) for j in range(-2, 3)]

In [28]:
# Derivatives
derivs = {}
for n in range(5):
    for m in range(5):
        name = f"u_{n}x_{m}y"
        sym = sp.Symbol(name, real=True)
        derivs[(n, m)] = sym

        # Populate Latex map: e.g. u_{xx, yyy}
        x_str = "x" * n
        y_str = "y" * m
        comma = "," if (n > 0 and m > 0) else ""
        if n == 0 and m == 0:
            symbol_names[sym] = "u"
        else:
            symbol_names[sym] = rf"u_{{{x_str}{comma}{y_str}}}"



In [29]:
for i, j in points:
    name = f"u_{str(i).replace('-','m')}_{str(j).replace('-','m')}"
    sym = sp.Symbol(name, real=True)
    u_dict[(i, j)] = sym

    i_str = get_index_str(i, 'i')
    j_str = get_index_str(j, 'j')
    symbol_names[sym] = rf"u_{{{i_str}, {j_str}}}"

LatexPrinter.set_global_settings(symbol_names=symbol_names)


def expand_to_taylor_2d(expr):
    expanded = expr
    for (di, dj), sym in u_dict.items():
        taylor_series = 0
        for n in range(5):
            for m in range(5):
                # Can be factored in taylor_spatial_2d
                taylor_series += (di * dx)**n / sp.factorial(n) * (
                    dj * dy)**m / sp.factorial(m) * derivs[(n, m)]
        expanded = expanded.subs(sym, taylor_series)
    return expanded

In [30]:
u_i = u_dict[(0, 0)]
# Dissipation on a cross stencil
dissipation = sigma * u_i + (1 - sigma) * (u_dict[(1,0)] + u_dict[(-1,0)] + u_dict[(0,1)] + u_dict[(0,-1)]) / 4
# First order advection
adv = -a * dt * (u_dict[(1,0)] - u_dict[(-1,0)]) / (2 * dx) \
      -b * dt * (u_dict[(0,1)] - u_dict[(0,-1)]) / (2 * dy)
# Second order diffusion (L-F)
diff_xx = (a * dt)**2 / 2 * (u_dict[(1,0)] - 2*u_i + u_dict[(-1,0)]) / dx**2
diff_yy = (b * dt)**2 / 2 * (u_dict[(0,1)] - 2*u_i + u_dict[(0,-1)]) / dy**2
diff_xy = 2 * a * b * dt**2 / 2 * (u_dict[(1,1)] - u_dict[(-1,1)] - u_dict[(1,-1)] + u_dict[(-1,-1)]) / (4 * dx * dy)

u_next = dissipation + adv + diff_xx + diff_yy + diff_xy

In [31]:
display(Math(r"u^{n+1}_{i, j} = " + sp.latex(sp.collect(u_next, [u_i, sigma, dt]))))

<IPython.core.display.Math object>

In [32]:
u_exact_taylor = 0
for k in range(5):
    term = 0
    for l in range(k + 1):
        if (l, k - l) in derivs:
            coeff = sp.binomial(k, l) * (-a)**l * (-b)**(k - l)
            term += coeff * derivs[(l, k - l)]
    u_exact_taylor += (dt**k / sp.factorial(k)) * term


# Truncation Error
u_next_taylor = expand_to_taylor_2d(u_next)
error = sp.expand(u_next_taylor - u_exact_taylor)

# Extract Error Coefficients
print("Extracting coefficients...")
error_coeffs = {}
for n in range(5):
    for m in range(5):
        coeff = sp.together(sp.simplify(sp.expand(error).coeff(derivs[(n, m)])))
        error_coeffs[(n, m)] = coeff

# IMPORTANT adjustment for 4th order
error_coeffs[(4, 0)] -= ((sigma - 1) * dx**4) / 24
error_coeffs[(0, 4)] -= ((sigma - 1) * dy**4) / 24

error_vector = sp.Matrix([error_coeffs[(n, m)] for n in range(5) for m in range(5)])

# Build the 25x25 Taylor Matrix
taylor_matrix = sp.Matrix([
    [ (di * dx)**n / sp.factorial(n) * (dj * dy)**m / sp.factorial(m) for (di, dj) in points ]
    for n in range(5) for m in range(5)
])

print("Solving system (may take a few seconds)")
solution = taylor_matrix.solve(error_vector)

Extracting coefficients...
Solving system (may take a few seconds)


In [33]:
stencil_vector = sp.Matrix([u_dict[p] for p in points])
correction_term = - solution.dot(stencil_vector)

u_corrected = u_next + sp.simplify(correction_term)

In [39]:
sp.collect(sp.expand(u_corrected), u_dict.values())

u_0_0*(a**4*dt**4/(4*dx**4) + 25*a**2*b**2*dt**4/(16*dx**2*dy**2) - 5*a**2*dt**2/(4*dx**2) + b**4*dt**4/(4*dy**4) - 5*b**2*dt**2/(4*dy**2) + sigma/2 + 1/2) + u_0_1*(-5*a**2*b**2*dt**4/(6*dx**2*dy**2) + 5*a**2*b*dt**3/(6*dx**2*dy) - b**4*dt**4/(6*dy**4) + b**3*dt**3/(6*dy**3) + 2*b**2*dt**2/(3*dy**2) - 2*b*dt/(3*dy) - sigma/6 + 1/6) + u_0_2*(5*a**2*b**2*dt**4/(96*dx**2*dy**2) - 5*a**2*b*dt**3/(48*dx**2*dy) + b**4*dt**4/(24*dy**4) - b**3*dt**3/(12*dy**3) - b**2*dt**2/(24*dy**2) + b*dt/(12*dy) + sigma/24 - 1/24) + u_0_m1*(-5*a**2*b**2*dt**4/(6*dx**2*dy**2) - 5*a**2*b*dt**3/(6*dx**2*dy) - b**4*dt**4/(6*dy**4) - b**3*dt**3/(6*dy**3) + 2*b**2*dt**2/(3*dy**2) + 2*b*dt/(3*dy) - sigma/6 + 1/6) + u_0_m2*(5*a**2*b**2*dt**4/(96*dx**2*dy**2) + 5*a**2*b*dt**3/(48*dx**2*dy) + b**4*dt**4/(24*dy**4) + b**3*dt**3/(12*dy**3) - b**2*dt**2/(24*dy**2) - b*dt/(12*dy) + sigma/24 - 1/24) + u_1_0*(-a**4*dt**4/(6*dx**4) + a**3*dt**3/(6*dx**3) - 5*a**2*b**2*dt**4/(6*dx**2*dy**2) + 2*a**2*dt**2/(3*dx**2) + 5*a*b**

In [35]:
error_corrected = sp.expand(expand_to_taylor_2d(u_corrected) - u_exact_taylor)
error_corrected = sp.simplify(error_corrected)

error_corrected

dx**4*sigma*u_4x_0y/24 - dx**4*u_4x_0y/24 + dy**4*sigma*u_0x_4y/24 - dy**4*u_0x_4y/24